# 03 · Ablation cấu hình — Qwen2.5-VL-3B trên ViTextVQA

Notebook này trả lời **hai câu hỏi cấu hình** phải chốt trước khi train QLoRA:

| Phần | Câu hỏi | Cách trả lời |
|---|---|---|
| **A — dtype probe** | 23 output hỏng `!!!!` ở W5 có phải do tràn số fp16 không? | Chạy lại đúng 23 câu đó + 50 câu đối chứng ở hai chế độ compute: **fp16** và **fp32** |
| **B — ablation độ phân giải** | `max_pixels` 512 hay 1024 token ảnh? 4-bit hay fp16? | Quét ma trận config trên **tập dev** (KHÔNG dùng test — tránh tuning trên tập báo cáo) |

**Nguyên tắc:** mọi biến thể chạy qua **cùng** `src/eval/run_baseline.py` — chỉ khác file config. Nhờ vậy so sánh đúng nghĩa *under identical conditions*, và mỗi biến thể tự sinh `metrics.json` có đủ seed / GPU / commit / sha256 dữ liệu.

**Điều kiện:** Runtime → Change runtime type → **T4 GPU**. Data ViTextVQA chính thức đã có trên Drive.

**Nếu Colab rớt:** chạy lại từ phần 1 — các lô đã xong tự bỏ qua (resume).

## 1. Thiết lập môi trường

Mount Drive, clone/pull code mới nhất, cài thư viện (pin `transformers==5.17.0` để nhất quán với baseline W5), nạp `HF_TOKEN` nếu có.

*Kỳ vọng: in commit (≥ `26af132`), GPU `Tesla T4`, `✅ Thiết lập xong`. ~1–2 phút.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU -> đổi runtime sang T4')
print('✅ Thiết lập xong')

## 2. Ghi nhận khả năng bf16 của GPU

**Vì sao cần cell này:** bf16 là dtype được Qwen2.5-VL khuyến nghị vì khoảng biểu diễn rộng như fp32 nên không tràn. Nhưng bf16 chỉ có từ kiến trúc Ampere (sm_80) trở lên — T4 là Turing (sm_75). Cell này **chứng minh bằng số** thay vì nói suông, và con số đó đi thẳng vào phần Methodology: *ràng buộc phần cứng buộc ta chọn giữa fp16 (nhanh, rủi ro tràn) và fp32 (an toàn, chậm hơn)*.

*Kỳ vọng trên T4: `compute capability (7, 5)`, `bf16 hỗ trợ: False`.*

In [ ]:
import torch, platform, json, os

gpu_name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
bf16_ok = torch.cuda.is_bf16_supported()
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

HW = {
    'gpu': gpu_name,
    'compute_capability': f'sm_{cap[0]}{cap[1]}',
    'bf16_supported': bool(bf16_ok),
    'vram_gb': round(vram_gb, 1),
    'torch': torch.__version__,
    'platform': platform.platform(),
}
print(json.dumps(HW, indent=2, ensure_ascii=False))

if not bf16_ok:
    print('\n→ GPU KHÔNG hỗ trợ bf16. Lựa chọn thu về: compute fp16 (rủi ro tràn) vs fp32 (an toàn, chậm hơn).')
else:
    print('\n→ GPU CÓ bf16. Cân nhắc thêm nhánh bf16 vào probe ở phần 5.')

## 3. Dựng tập probe: 23 câu hỏng + 50 câu đối chứng

**Vấn đề:** nếu chỉ chạy lại 23 câu hỏng, ta không biết config mới có **làm hỏng những câu vốn đang đúng** hay không. Vì vậy phải kèm một **tập đối chứng (control set)** — 50 câu bình thường lấy ngẫu nhiên seed 42 từ chính tập test 2.000 câu, loại trừ 23 câu hỏng.

Tập probe được ghi thành **file dữ liệu thật trên Drive** (không phải biến trong RAM) để `run_baseline.py` tính được `sha256` của nó → tái lập được.

*Kỳ vọng: `23 hỏng + 50 đối chứng = 73 mẫu`, in đường dẫn file probe.*

In [ ]:
import json, os, random

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DATA_DIR   = f'{DRIVE_ROOT}/data/vitextvqa_official'
EXP_ROOT   = f'{DRIVE_ROOT}/experiments'
SEED       = 42
N_CONTROL  = 50

# 1) Danh sách 23 câu hỏng (sinh ở W6 bởi src/eval/find_degraded.py, đã commit)
failed = json.load(open('experiments/W06_dtype_probe_seed42/failed_ids.json', encoding='utf-8'))
failed_ids = set(failed['question_ids'])
print(f"Câu hỏng: {failed['num_degraded']}/{failed['num_total']} (nguồn sha256 {failed['source_sha256'][:12]}…)")

# 2) Tập test 2.000 câu gốc của W5
test_path = f'{DATA_DIR}/test_subset2000_seed42.json'
test = json.load(open(test_path, encoding='utf-8'))
assert len(test) == 2000, f'Kỳ vọng 2000 mẫu, nhận {len(test)}'

# 3) Bốc 50 câu đối chứng (loại trừ các câu hỏng), seed cố định
pool = [r for r in test if str(r['question_id']) not in failed_ids]
control_ids = set(str(r['question_id']) for r in random.Random(SEED).sample(pool, N_CONTROL))

# 4) Giữ NGUYÊN thứ tự file gốc -> hai lần dựng lại cho ra file giống hệt nhau
keep = failed_ids | control_ids
probe = [r for r in test if str(r['question_id']) in keep]

PROBE_PATH = f'{DATA_DIR}/probe_dtype{len(probe)}_seed{SEED}.json'
with open(PROBE_PATH, 'w', encoding='utf-8') as fp:
    json.dump(probe, fp, ensure_ascii=False)

print(f'Probe: {len(failed_ids)} hỏng + {len(control_ids)} đối chứng = {len(probe)} mẫu')
print('Ghi ->', PROBE_PATH)
assert len(probe) == len(failed_ids) + N_CONTROL

## 4. Hàm chạy một biến thể config

**Ý tưởng:** không sửa `configs/qwen_lora.yaml` gốc (nó là bản chuẩn W5, sửa vào là mất mốc so sánh). Thay vào đó mỗi biến thể **sinh một file config tạm** trong `/content/cfg_variants/`, rồi gọi `run_baseline.run(..., check_config=False)`.

Ba hàm phụ trợ:
- `make_cfg(...)` — đẻ file config biến thể.
- `run_variant(...)` — chạy + bắt lỗi OOM (hết VRAM) thay vì để notebook chết giữa chừng.
- `count_degraded(...)` — đếm output hỏng, **dùng đúng hàm `is_degraded` đã commit** (không viết lại logic ở hai nơi).

*Kỳ vọng: chỉ in `✅ Hàm đã sẵn sàng.`, chưa chạy GPU.*

In [ ]:
import copy, gc, time, yaml, torch
from src.eval import run_baseline
from src.eval.find_degraded import is_degraded

BASE_CFG = 'configs/qwen_lora.yaml'
CFG_DIR  = '/content/cfg_variants'
os.makedirs(CFG_DIR, exist_ok=True)

def make_cfg(name, compute_dtype='float16', torch_dtype='float16',
             max_pixels=401408, load_in_4bit=True):
    """Sinh file config cho 1 biến thể. Trả đường dẫn file."""
    cfg = yaml.safe_load(open(BASE_CFG, encoding='utf-8'))
    cfg['model']['torch_dtype'] = torch_dtype       # dtype của phần KHÔNG lượng tử hoá (vision tower)
    cfg['model']['max_pixels']  = max_pixels
    cfg['quantization']['enabled']                 = load_in_4bit
    cfg['quantization']['load_in_4bit']            = load_in_4bit
    cfg['quantization']['bnb_4bit_compute_dtype']  = compute_dtype
    path = f'{CFG_DIR}/{name}.yaml'
    with open(path, 'w', encoding='utf-8') as fp:
        yaml.safe_dump(cfg, fp, allow_unicode=True)
    return path

def count_degraded(run_name):
    """Đọc predictions.json của 1 run -> (số câu hỏng, danh sách id)."""
    preds = json.load(open(f'{EXP_ROOT}/{run_name}/predictions.json', encoding='utf-8'))
    bad = [r['question_id'] for r in preds if is_degraded(r['prediction'])]
    return len(bad), bad

def run_variant(run_name, data_file, chunk=25, **cfg_kw):
    """Chạy 1 biến thể; trả dict kết quả gọn. Hết VRAM -> ghi nhận, không làm chết notebook."""
    cfg_path = make_cfg(run_name, **cfg_kw)
    print(f'\n{"="*70}\n▶ {run_name}  |  {cfg_kw}\n{"="*70}')
    t0 = time.time()
    try:
        res = run_baseline.run(
            dataset='vitextvqa_official', run_name=run_name,
            drive_root=DRIVE_ROOT, data_file=data_file,
            out_root=EXP_ROOT, config=cfg_path,
            chunk=chunk, check_config=False,
        )
        n_bad, bad_ids = count_degraded(run_name)
        out = {'run_name': run_name, 'status': 'ok', **cfg_kw,
               'exact_match': round(res['metrics']['exact_match'], 4),
               'anls':        round(res['metrics']['anls'], 4),
               'n_degraded':  n_bad,
               'degraded_ids': bad_ids,
               'sec_per_sample': res['meta']['sec_per_sample'],
               'elapsed_sec': round(time.time() - t0, 1)}
    except torch.cuda.OutOfMemoryError as exc:
        out = {'run_name': run_name, 'status': 'OOM', **cfg_kw, 'error': str(exc)[:200]}
        print('⚠️ HẾT VRAM — bỏ qua biến thể này, ghi nhận vào bảng.')
    finally:
        gc.collect(); torch.cuda.empty_cache()
    print('→', {k: v for k, v in out.items() if k != 'degraded_ids'})
    return out

RESULTS = []
print('✅ Hàm đã sẵn sàng.')

## 5. Nhánh A — tái hiện lỗi với fp16 (config y hệt W5)

**Đây là cell quan trọng nhất và tuyệt đối không được bỏ qua.** Nó trả lời câu hỏi nền: *lỗi có tái lập được không?*

- Nếu ra **đúng 23 câu hỏng** → lỗi mang tính hệ thống, do config, không phải ngẫu nhiên. Lúc đó nhánh B mới có nghĩa.
- Nếu ra **ít hơn 23** → kết quả W5 không tái lập được. Đó là vấn đề nghiêm trọng hơn nhiều (seed/non-determinism) và phải điều tra trước khi làm bất cứ việc gì khác.

*Kỳ vọng: 73 mẫu, ~2 phút chạy (+ ~4 phút nạp model lần đầu), `n_degraded = 23`.*

In [ ]:
RESULTS.append(run_variant(
    run_name='W06_probe_fp16_qwen25vl_vitextvqa73_seed42',
    data_file=PROBE_PATH,
    compute_dtype='float16', torch_dtype='float16', max_pixels=401408,
))

## 6. Nhánh B — đổi compute sang fp32

Chỉ đổi **đúng một thứ**: kiểu số dùng khi tính toán (`float16` → `float32`), áp cho cả phần lượng tử hoá (`bnb_4bit_compute_dtype`) lẫn phần không lượng tử hoá (`torch_dtype`, chủ yếu là vision encoder — nơi thường tràn nhất với ảnh độ phân giải cao).

**Weight vẫn nén 4-bit NF4** — nghĩa là ràng buộc PEFT/efficiency không hề bị nới. Chỉ tốc độ chậm đi.

*Kỳ vọng: ~3–5 phút, `n_degraded = 0` nếu giả thuyết tràn số đúng.*

In [ ]:
RESULTS.append(run_variant(
    run_name='W06_probe_fp32_qwen25vl_vitextvqa73_seed42',
    data_file=PROBE_PATH,
    compute_dtype='float32', torch_dtype='float32', max_pixels=401408,
))

## 7. Kết luận phần A

Bảng so sánh + **kết luận tự động** theo đúng ba kịch bản đã định trước (không nhìn số rồi mới nghĩ ra lời giải thích — đó là p-hacking).

Cell cũng ghi `probe_summary.json` để commit làm bằng chứng.

*Kỳ vọng: bảng 2 dòng + một dòng KẾT LUẬN rõ ràng.*

In [ ]:
import pandas as pd

df = pd.DataFrame([{k: v for k, v in r.items() if k != 'degraded_ids'} for r in RESULTS])
display(df)

fp16 = next((r for r in RESULTS if 'probe_fp16' in r['run_name']), None)
fp32 = next((r for r in RESULTS if 'probe_fp32' in r['run_name']), None)

verdict, decision = 'CHƯA ĐỦ DỮ LIỆU', '—'
if fp16 and fp32 and fp16['status'] == 'ok' and fp32['status'] == 'ok':
    if fp16['n_degraded'] == 23 and fp32['n_degraded'] == 0:
        verdict  = 'XÁC NHẬN: tràn số fp16 là nguyên nhân của 23 output hỏng.'
        decision = 'Chốt compute_dtype=float32 cho baseline + QLoRA -> PHẢI chạy lại baseline với config mới.'
    elif fp16['n_degraded'] == 23 and fp32['n_degraded'] > 0:
        verdict  = 'BÁC BỎ: đổi dtype không sửa được -> nguyên nhân khác (nghi ảnh/prompt).'
        decision = 'Điều tra tiếp: log độ phân giải ảnh + độ dài prompt của các câu còn hỏng.'
    elif fp16['n_degraded'] != 23:
        verdict  = f"CẢNH BÁO: fp16 chỉ tái hiện {fp16['n_degraded']}/23 -> kết quả W5 KHÔNG tái lập được."
        decision = 'Dừng ablation. Truy nguồn non-determinism (seed, phiên bản thư viện, GPU) trước.'

print('\n' + '='*70)
print('KẾT LUẬN :', verdict)
print('QUYẾT ĐỊNH:', decision)
if fp16 and fp32 and fp16['status'] == fp32['status'] == 'ok':
    slow = fp32['sec_per_sample'] / max(fp16['sec_per_sample'], 1e-9)
    print(f"CHI PHÍ   : fp32 chậm hơn fp16 {slow:.2f}× ({fp16['sec_per_sample']}s -> {fp32['sec_per_sample']}s mỗi câu)")
print('='*70)

OUT_A = f'{EXP_ROOT}/W06_dtype_probe_seed42'
os.makedirs(OUT_A, exist_ok=True)
with open(f'{OUT_A}/probe_summary.json', 'w', encoding='utf-8') as fp:
    json.dump({'hardware': HW, 'probe_file': os.path.basename(PROBE_PATH),
               'n_probe': len(probe), 'n_failed_source': failed['num_degraded'],
               'verdict': verdict, 'decision': decision, 'runs': RESULTS},
              fp, ensure_ascii=False, indent=2)
print('Ghi ->', f'{OUT_A}/probe_summary.json')

## 8. Phần B — dựng tập dev cho ablation độ phân giải

**Vì sao phải dùng dev, không dùng test:** test là tập dùng để **báo cáo** con số cuối. Nếu ta thử 4 config trên test rồi chọn config tốt nhất, con số báo cáo đã bị *tuned* trên chính tập đó — hội đồng có quyền bác bỏ. Dev tồn tại đúng để làm việc này.

ViTextVQA chính thức có `dev.json` (5.155 câu). Ta bốc **300 câu seed 42** cho vòng lặp nhanh.

*Kỳ vọng: in `300 mẫu`, đường dẫn file dev subset.*

In [ ]:
dev_path = f'{DATA_DIR}/dev.json'
assert os.path.exists(dev_path), f'Không thấy {dev_path} -> chạy lại src/data/prepare_vitextvqa.py'
dev = json.load(open(dev_path, encoding='utf-8'))
print('dev.json:', len(dev), 'câu')

N_DEV = 300
dev_ids = set(str(r['question_id']) for r in random.Random(SEED).sample(dev, N_DEV))
dev_sub = [r for r in dev if str(r['question_id']) in dev_ids]   # giữ thứ tự gốc

DEV_PATH = f'{DATA_DIR}/dev_subset{len(dev_sub)}_seed{SEED}.json'
with open(DEV_PATH, 'w', encoding='utf-8') as fp:
    json.dump(dev_sub, fp, ensure_ascii=False)
print(f'{len(dev_sub)} mẫu -> {DEV_PATH}')

## 9. Ma trận ablation trên dev

Hai biến thể, chỉ đổi **một** chiều (độ phân giải) để quy trách nhiệm được:

| Mã | `max_pixels` | Lượng tử hoá | Câu hỏi trả lời |
|---|---|---|---|
| A1 | 401408 (512 token) | 4-bit NF4 | Mốc — chính là config W5 |
| A2 | 1003520 (1280 token) | 4-bit NF4 | Độ phân giải cao có cứu được scene-text không? |

A3/A4 (không lượng tử hoá) bị loại: weight fp32 ~12,4 GB, chắc chắn OOM trên T4 15GB; hoãn sang W7 trên L4.

`DTYPE` lấy theo **kết luận phần A** — nếu fp32 thắng thì cả 2 biến thể chạy fp32 để không trộn hai biến.

*Kỳ vọng: ~40–50 phút cho cả 2 biến thể. Giữ tab mở.*

In [ ]:
# Lấy dtype theo kết luận phần A (sửa tay nếu muốn ép khác)
DTYPE = 'float32' if (fp32 and fp32.get('status') == 'ok' and fp32.get('n_degraded', 1) == 0) else 'float16'
print('dtype dùng cho ablation:', DTYPE)

MATRIX = [
    ('A1_mp512_4bit',  401408,  True),
    ('A2_mp1280_4bit', 1003520, True),
]

ABLATION = []
for tag, mp, q4 in MATRIX:
    ABLATION.append(run_variant(
        run_name=f'W06_ablation_{tag}_qwen25vl_vitextvqa_dev300_seed42',
        data_file=DEV_PATH, chunk=100,
        compute_dtype=DTYPE, torch_dtype=DTYPE,
        max_pixels=mp, load_in_4bit=q4,
    ))

## 10. Bảng tổng hợp + chốt config

In bảng 4 dòng và ghi `ablation_summary.json`.

**Cách đọc bảng — đừng chỉ nhìn EM cao nhất:** nếu A2 hơn A1 vài điểm nhưng chậm gấp đôi, cân nhắc giữ A1, vì *efficiency là ràng buộc thiết kế tường minh* của đề tài. Một config đắt hơn phải **đáng giá bằng số**, không phải bằng cảm giác.

*Kỳ vọng: bảng có cột EM / ANLS / s-mỗi-câu / số câu hỏng.*

In [ ]:
cols = ['run_name', 'status', 'max_pixels', 'load_in_4bit', 'compute_dtype',
        'exact_match', 'anls', 'sec_per_sample', 'n_degraded', 'elapsed_sec']
dfa = pd.DataFrame([{k: r.get(k) for k in cols} for r in ABLATION])
display(dfa)

ok = [r for r in ABLATION if r.get('status') == 'ok']
if ok:
    best = max(ok, key=lambda r: r['exact_match'])
    base = next((r for r in ok if 'A1_' in r['run_name']), None)
    print(f"\nEM cao nhất: {best['run_name']} -> EM {best['exact_match']} | {best['sec_per_sample']}s/câu")
    if base and best is not base:
        d_em   = best['exact_match'] - base['exact_match']
        d_cost = best['sec_per_sample'] / max(base['sec_per_sample'], 1e-9)
        print(f"So với A1 (config W5): {d_em:+.4f} EM, chi phí {d_cost:.2f}×")
        print('-> Chỉ đổi config nếu mức tăng EM xứng với chi phí; ghi rõ lý do vào R4.')

OUT_B = f'{EXP_ROOT}/W06_ablation_config_dev300_seed42'
os.makedirs(OUT_B, exist_ok=True)
with open(f'{OUT_B}/ablation_summary.json', 'w', encoding='utf-8') as fp:
    json.dump({'hardware': HW, 'dev_file': os.path.basename(DEV_PATH), 'n_dev': len(dev_sub),
               'dtype': DTYPE, 'runs': ABLATION}, fp, ensure_ascii=False, indent=2)
print('Ghi ->', f'{OUT_B}/ablation_summary.json')

## 11. Bước tiếp theo

Hai file tóm tắt đã nằm trên Drive:

```
experiments/W06_dtype_probe_seed42/probe_summary.json
experiments/W06_ablation_config_dev300_seed42/ablation_summary.json
```

Dán nội dung hai file (hoặc ảnh chụp hai bảng) cho Claude để:
1. Nghiệm thu kết luận phần A và quyết định có phải chạy lại baseline không.
2. Chốt config cuối cho `configs/qwen_lora.yaml` (việc 3 của W6).
3. Sinh prompt Cursor chép kết quả từ ổ `G:` vào repo + commit.

**Chưa sửa `configs/qwen_lora.yaml` ở bước này** — config chuẩn chỉ đổi sau khi đã có kết luận và ghi lý do vào Methodology.